In [1]:
import os
import networkx as nx
import matplotlib.pyplot as plt
import time
from collections import Counter
import numpy as np

import scipy
import pandas as pd
import json

In [2]:
network_id = 'cen'
resolution = '.001'

lfr_dir = f'data/networks/{network_id}_lfr_networks/{network_id}_leiden{resolution}_lfr'

abcd_dir = f'data/networks/{network_id}_abcd_networks/{network_id}_leiden{resolution}_abcd'
if not os.path.exists(abcd_dir):
    os.system(f'mkdir -p {abcd_dir}')

# **Process statistics**

In [3]:
def gen_lfr(stats_path, cmin):
    with open(stats_path) as f:
        net_cluster_stats = json.load(f)
    
    if int(cmin) > net_cluster_stats['max-cluster-size']:
        return
    
    if net_cluster_stats['node-count'] > 5000000:
        ratio = net_cluster_stats['node-count'] / 3000000
        net_cluster_stats['node-count'] = 3000000
        net_cluster_stats['max-degree'] = int(net_cluster_stats['max-degree'] / ratio)
        net_cluster_stats['max-cluster-size'] = int(net_cluster_stats['max-cluster-size'] / ratio)
        net_cluster_stats['max-cluster-size'] = 1000
    
    if net_cluster_stats['mean-degree'] < 4:
        net_cluster_stats['max-degree'] = 31
    
    if net_cluster_stats['max-degree'] > 1000:
       net_cluster_stats['max-degree'] = 1000
    
    if net_cluster_stats['max-cluster-size'] > 5000:
       net_cluster_stats['max-cluster-size'] = 5000
    
    if net_cluster_stats['mean-degree'] > 50:
       net_cluster_stats['max-cluster-size'] = 1000
    
    N = net_cluster_stats['node-count']
    k = net_cluster_stats['mean-degree']
    mink = net_cluster_stats['min-degree']
    maxk = net_cluster_stats['max-degree']
    mu = net_cluster_stats['mixing-parameter']
    maxc = net_cluster_stats['max-cluster-size']
    minc = int(cmin)
    t1 = net_cluster_stats['tau1']
    t2 = net_cluster_stats['tau2']
    
    return N, k, mink, maxk, mu, maxc, minc, t1, t2

In [4]:
network_stats_json_path = f'data/network_params/{network_id}_leiden{resolution}.json'
minc = 9
N, k, mink, maxk, mu, maxc, minc, t1, t2 = gen_lfr(network_stats_json_path, minc)

# **LFR**

In [5]:
base_path = os.getcwd()

cmd = base_path + '/package1/binary_networks/benchmark' \
        + ' -N ' + str(N) \
        + ' -k ' + str(k) \
        + ' -maxk ' + str(maxk) \
        + ' -mu ' + str(mu) \
        + ' -maxc ' + str(maxc) \
        + ' -minc ' + str(minc) \
        + ' -t1 ' + str(t1) \
        + ' -t2 ' + str(t2)
        
print(cmd)

/home/vltanh/synnet/package1/binary_networks/benchmark -N 3000000 -k 13.1600803635 -maxk 1000 -mu 0.5216179061 -maxc 1000 -minc 9 -t1 2.6183156732 -t2 2.3625931615


In [6]:
lfr_net_dir = 'data/output/' + os.path.basename(network_stats_json_path).replace('.json', '') + "_lfr"
if not os.path.exists(lfr_net_dir):
    os.system('mkdir -p ' + lfr_net_dir)

In [7]:
os.chdir(lfr_net_dir)
try:
    start = time.perf_counter()
    os.system(cmd)
    elapsed = time.perf_counter() - start
    print(f"Generation time: {elapsed}")
finally:
    os.chdir(base_path)

Generation time: 0.03236391022801399


/home/vltanh/synnet/package1/binary_networks/benchmark: /lib64/libstdc++.so.6: version `CXXABI_1.3.9' not found (required by /home/vltanh/synnet/package1/binary_networks/benchmark)
/home/vltanh/synnet/package1/binary_networks/benchmark: /lib64/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /home/vltanh/synnet/package1/binary_networks/benchmark)
/home/vltanh/synnet/package1/binary_networks/benchmark: /lib64/libstdc++.so.6: version `GLIBCXX_3.4.21' not found (required by /home/vltanh/synnet/package1/binary_networks/benchmark)


# Compute LFR stats

In [14]:
os.system(f'python cluster-statistics/stats.py -i {lfr_dir}/network.dat -e {lfr_dir}/community.dat -o {lfr_dir}/lfr_stats.csv')

Loading clusters...
Done
Loading graph...
Done
Computing modularity...
Done
Realizing clusters...
Done
Computing mincut...
Done
Computing conductance...


In [ ]:
lfr_stats = pd.read_csv(f'{lfr_net_dir}/lfr_stats.csv')
c = Counter(lfr_stats['connectivity'].tolist())
plt.bar(c.keys(), c.values())

In [ ]:
EPS = 1e-8
bins = np.arange(0, 4 + EPS, 0.1) + EPS
bins[0] -= EPS
print(bins)

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
ax.hist(lfr_stats['connectivity_normalized_log10(n)'].tolist(), bins=bins)
# ax[1].hist(lfr_stats['connectivity_normalized_log2(n)'].tolist())
# ax[2].hist(lfr_stats['connectivity_normalized_sqrt(n)/5'].tolist())
ax.set_xlabel('mincut / log10(n)')
ax.set_ylabel('count')
ax.axvline(1.0, color='red')
fig.tight_layout()
fig.savefig(f'{lfr_net_dir}/lfr.pdf')

wellconnected = len([x for x in lfr_stats['connectivity_normalized_log10(n)'].tolist() if x > 1.0])
print(f'Well connected: {wellconnected} / {len(lfr_stats) - 1} = {wellconnected / (len(lfr_stats) - 1)}')

disconnected = len([x for x in lfr_stats['connectivity_normalized_log10(n)'].tolist() if x < EPS])
print(f'Well connected: {disconnected} / {len(lfr_stats) - 1} = {disconnected / (len(lfr_stats) - 1)}')

In [ ]:
os.system(f'python cluster-statistics/summarize.py {lfr_net_dir}/lfr_stats.csv {lfr_net_dir}/network.dat')

In [ ]:
lfr_summary = pd.read_csv(f'{lfr_net_dir}/lfr_stats_summary.csv', header=None)
for row in lfr_summary.iterrows():
    k, v = row[1].values
    print(f'{k} \t\t\t {v}')

In [ ]:
# Read generated LFR network
G = nx.read_edgelist([' '.join(x.strip().split('\t')) for x in open(f'{lfr_dir}/network.dat').readlines()], nodetype=int)

In [ ]:
# Generate degree sequence
degree = []
for u in G.nodes:
    degree.append(len(G[u]))
degree = sorted(degree, reverse=True)
    
with open(f'{abcd_dir}/deg.dat', 'w') as f:
    f.write('\n'.join(map(str, degree)))
    
plt.hist(degree, bins=10)
scipy.stats.describe(degree)

In [ ]:
# Generate community size sequence
cs = {}
for line in open(f'{lfr_dir}/community.dat').readlines():
    node, comm = map(int, line.strip().split('\t'))
    cs.setdefault(comm, 0)
    cs[comm] += 1
cs = sorted(cs.values(), reverse=True)

with open(f'{abcd_dir}/cs.dat', 'w') as f:
    f.write('\n'.join(map(str, cs)))
    
plt.hist(cs)
scipy.stats.describe(cs)

# **ABCD** from LFR

In [10]:
seed = 0
xi = mu

In [11]:
start = time.perf_counter()
os.system(f'~/.juliaup/bin/julia ABCDGraphGenerator.jl/utils/graph_sampler.jl {abcd_dir}/edge.dat {abcd_dir}/com.dat {abcd_dir}/deg.dat {abcd_dir}/cs.dat xi {xi} false false {seed} 0')
elapsed = time.perf_counter() - start
print(f"Generation time: {elapsed}")

ERROR: LoadError: 

Generation time: 4.37528816703707


ArgumentError: Package ABCDGraphGenerator not found in current path.
- Run `import Pkg; Pkg.add("ABCDGraphGenerator")` to install the ABCDGraphGenerator package.
Stacktrace:
 [1] macro expansion
   @ ./loading.jl:1772 [inlined]
 [2] macro expansion
   @ ./lock.jl:267 [inlined]
 [3] __require(into::Module, mod::Symbol)
   @ Base ./loading.jl:1753
 [4] #invoke_in_world#3
   @ ./essentials.jl:926 [inlined]
 [5] invoke_in_world
   @ ./essentials.jl:923 [inlined]
 [6] require(into::Module, mod::Symbol)
   @ Base ./loading.jl:1746
in expression starting at /home/vltanh/synnet/ABCDGraphGenerator.jl/utils/graph_sampler.jl:1


In [8]:
start = time.perf_counter()
os.system(f'julia ABCDGraphGenerator.jl/utils/graph_sampler.jl {abcd_dir}/edge.dat {abcd_dir}/com.dat {abcd_dir}/deg.dat {abcd_dir}/cs.dat xi {xi} false false {seed} 0')
elapsed = time.perf_counter() - start
print(f"Generation time: {elapsed}")

Generation time: 0.015944366110488772


sh: julia: command not found


# Compute ABCD stats

In [ ]:
os.system(f'python cluster-statistics/stats.py -i {lfr_net_dir}/abcd_edge.dat -e {lfr_net_dir}/abcd_com.dat -o {lfr_net_dir}/abcd_stats.csv')

In [ ]:
abcd_stats = pd.read_csv(f'{lfr_net_dir}/abcd_stats.csv')
c = Counter(abcd_stats['connectivity'].tolist())
plt.bar(c.keys(), c.values())

In [ ]:
EPS = 1e-8
bins = np.arange(0, 3 + EPS, 0.1) + EPS
bins[0] -= EPS

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
ax.hist(abcd_stats['connectivity_normalized_log10(n)'].tolist(), bins=bins)
# ax[1].hist(abcd_stats['connectivity_normalized_log2(n)'].tolist())
# ax[2].hist(abcd_stats['connectivity_normalized_sqrt(n)/5'].tolist())
ax.set_xlabel('mincut / log10(n)')
ax.set_ylabel('count')
ax.axvline(1.0, color='red')
fig.tight_layout()
fig.savefig(f'{lfr_net_dir}/abcd.pdf')

wellconnected = len([x for x in abcd_stats['connectivity_normalized_log10(n)'].tolist() if x > 1.0])
print(f'Well connected: {wellconnected} / {len(abcd_stats) - 1} = {wellconnected / (len(abcd_stats) - 1)}')

disconnected = len([x for x in abcd_stats['connectivity_normalized_log10(n)'].tolist() if x < EPS])
print(f'Well connected: {disconnected} / {len(abcd_stats) - 1} = {disconnected / (len(abcd_stats) - 1)}')

In [ ]:
os.system(f'python cluster-statistics/summarize.py {lfr_net_dir}/abcd_stats.csv {lfr_net_dir}/abcd_edge.dat')

In [ ]:
lfr_summary = pd.read_csv(f'{lfr_net_dir}/abcd_stats_summary.csv', header=None)
for row in lfr_summary.iterrows():
    k, v = row[1].values
    print(f'{k} \t\t\t {v}')

In [ ]:
# Read generated ABCD network
abcd_G = nx.read_edgelist([' '.join(x.strip().split('\t')) for x in open(f'{lfr_net_dir}/abcd_edge.dat').readlines()], nodetype=int)

In [ ]:
# Generate degree sequence
degree = []
for u in abcd_G.nodes:
    degree.append(len(abcd_G[u]))
degree = sorted(degree, reverse=True)
    
plt.hist(degree, bins=10)
scipy.stats.describe(degree)

In [ ]:
# Generate community size sequence
cs = {}
for line in open(f'{lfr_net_dir}/abcd_com.dat').readlines():
    node, comm = map(int, line.strip().split('\t'))
    cs.setdefault(comm, 0)
    cs[comm] += 1
cs = sorted(cs.values(), reverse=True)
    
plt.hist(cs)
scipy.stats.describe(cs)

# **ABCD** from scratch

In [ ]:
with open(f'{lfr_net_dir}/abcds_config.toml', 'w') as f:
    f.write(
        f'''seed = ""                   # RNG seed, use "" for no seeding
n = "{N}"                   # number of vertices in graph
t1 = "{t1}"                      # power-law exponent for degree distribution
d_min = "{mink}"                   # minimum degree
d_max = "{maxk}"                  # maximum degree
d_max_iter = "{1000}"           # maximum number of iterations for sampling degrees
t2 = "{t2}"                      # power-law exponent for cluster size distribution
c_min = "{minc}"                  # minimum cluster size
c_max = "{maxc}"                # maximum cluster size
c_max_iter = "{1000}"           # maximum number of iterations for sampling cluster sizes
# Exactly one of xi and mu must be passed as Float64. Also if xi is provided islocal must be set to false or omitted.
xi = "{xi}"                    # fraction of edges to fall in background graph
#mu = "0.2"                   # mixing parameter
islocal = "false"             # if "true" mixing parameter is restricted to local cluster, otherwise it is global
isCL = "false"                # if "false" use configuration model, if "true" use Chung-Lu
degreefile = "{lfr_net_dir}/abcds_deg.dat"        # name of file do generate that contains vertex degrees
communitysizesfile = "{lfr_net_dir}/abcds_cs.dat" # name of file do generate that contains community sizes
communityfile = "{lfr_net_dir}/abcds_com.dat"     # name of file do generate that contains assignments of vertices to communities
networkfile = "{lfr_net_dir}/abcds_edge.dat"      # name of file do generate that contains edges of the generated graph
nout = "0"                  # number of vertices in graph that are outliers; optional parameter
                            # if nout is passed and is not zero then we require islocal = "false",
                            # isCL = "false", and xi (not mu) must be passed
                            # if nout > 0 then it is recommended that xi > 0'''        
    )

In [ ]:
start = time.perf_counter()
os.system(f'julia ABCDGraphGenerator.jl/utils/abcd_sampler.jl {lfr_net_dir}/abcds_config.toml')
elapsed = time.perf_counter() - start
print(f"Generation time: {elapsed}")

In [ ]:
os.system(f'python cluster-statistics/stats.py -i {lfr_net_dir}/abcds_edge.dat -e {lfr_net_dir}/abcds_com.dat -o {lfr_net_dir}/abcds_stats.csv')

In [ ]:
abcds_stats = pd.read_csv(f'{lfr_net_dir}/abcds_stats.csv')
c = Counter(abcds_stats['connectivity'].tolist())
plt.bar(c.keys(), c.values())

In [ ]:
EPS = 1e-8
bins = np.arange(0, 3 + EPS, 0.1) + EPS
bins[0] -= EPS

fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=300)
ax.hist(abcds_stats['connectivity_normalized_log10(n)'].tolist(), bins=bins)
# ax[1].hist(abcds_stats['connectivity_normalized_log2(n)'].tolist())
# ax[2].hist(abcds_stats['connectivity_normalized_sqrt(n)/5'].tolist())
ax.set_xlabel('mincut / log10(n)')
ax.set_ylabel('count')
ax.axvline(1.0, color='red')
fig.tight_layout()
fig.savefig(f'{lfr_net_dir}/abcds.pdf')

wellconnected = len([x for x in abcds_stats['connectivity_normalized_log10(n)'].tolist() if x > 1.0])
print(f'Well connected: {wellconnected} / {len(abcds_stats) - 1} = {wellconnected / (len(abcds_stats) - 1)}')

disconnected = len([x for x in abcds_stats['connectivity_normalized_log10(n)'].tolist() if x < EPS])
print(f'Disconnected: {disconnected} / {len(abcds_stats) - 1} = {disconnected / (len(abcds_stats) - 1)}')

# denseconnected = len([x for x in abcds_stats['connectivity_normalized_log10(n)'].tolist() if x > 3])
# print(f'Well connected: {denseconnected} / {len(abcds_stats) - 1} = {denseconnected / (len(abcds_stats) - 1)}')

In [ ]:
os.system(f'python cluster-statistics/summarize.py {lfr_net_dir}/abcds_stats.csv {lfr_net_dir}/abcds_edge.dat')

In [ ]:
lfr_summary = pd.read_csv(f'{lfr_net_dir}/abcds_stats_summary.csv', header=None)
for row in lfr_summary.iterrows():
    k, v = row[1].values
    print(f'{k} \t\t\t {v}')

In [ ]:
# Read generated ABCD network
abcds_G = nx.read_edgelist([' '.join(x.strip().split('\t')) for x in open(f'{lfr_net_dir}/abcds_edge.dat').readlines()], nodetype=int)

In [ ]:
# Generate degree sequence
degree = []
for u in abcd_G.nodes:
    degree.append(len(abcds_G[u]))
degree = sorted(degree, reverse=True)
    
plt.hist(degree, bins=10)
scipy.stats.describe(degree)

In [ ]:
# Generate community size sequence
cs = {}
for line in open(f'{lfr_net_dir}/abcds_com.dat').readlines():
    node, comm = map(int, line.strip().split('\t'))
    cs.setdefault(comm, 0)
    cs[comm] += 1
cs = sorted(cs.values(), reverse=True)
    
plt.hist(cs)
scipy.stats.describe(cs)